# Notebook 02 — Beat Averaging and Clustering Across All Animals

**Project:** Reproducible Signal Processing and Multivariate Analysis of Preclinical Mouse ECG Data for Cardiotoxicity Detection  
**Author:** Vishvasundar (MSc, University College Cork)  
**Supervisors:** Dr Luke Kelly (Statistics), Dr Roisin Kelly-Laubscher (Pharmacology)

## What this notebook does

Process every `.txt` ECG export in `data/`, isolate each animal's clean baseline window, detect R peaks, build an averaged-beat template, extract morphology features, and cluster the animals. Final outputs: `outputs/all_animals_features.csv` and four PNG figures.

### Annotation rule (Fix 1)
Annotations are matched generously by *prefix character* (`#*`, `#3`, `#1`) and the **gap between paired markers must be 10–120 s**. If the only candidate pair is too long (e.g. a 15-minute baseline), the first 60 s after the start marker is used. If only a single marker exists, a 30 s window after it is used. Non-baseline procedure markers (containing `oc`, `jugular`, `jugcan`) are excluded.

### ECG column rule (Fix 2)
The LabChart header tells us how many channels exist and which is *Channel 3*. The number of leading time columns is computed dynamically from `n_cols_in_first_data_row − n_channels`. Channel 3 is always the ECG. A sanity check rejects any column whose values look like BPM (300–700) instead of mV (±15).

### Robust R-peak detection (Fix 3)
`height = median + 4 × MAD` (median-absolute-deviation), so big artifact spikes do not raise the threshold above the real ~0.2 mV R peaks. If the resulting heart rate is outside 300–700 bpm we retry on the inverted signal. Files still outside 300–700 bpm are flagged `NEEDS_REVIEW` and kept in the audit list.

### QTc formula
**Mitchell:** `QTc = QT / √(RR/100)`, with QT and RR in milliseconds. This is the mouse-specific correction confirmed by Roisin. The output CSV carries an explicit `qtc_formula` column (value `Mitchell`) documenting which formula was applied.

> ⚠️ **OPEN ISSUE — QT measurement method differs between Notebook 01 and Notebook 02 (do not resolve yet; needs supervisor input).**
>
> The **QTc formula is identical** in both notebooks, but the **QT input differs** because of how each notebook measures the QT interval:
> - **Notebook 01:** QT measured **per beat** on the raw filtered baseline signal, then averaged → QT ≈ **70.5 ms**, QTc ≈ **63.9 ms** for Animal 201.
> - **Notebook 02 (this notebook):** QT measured on the **4-beat averaged template** (return-to-baseline within a fixed window) → QT = **40.0 ms**, QTc ≈ **36.3 ms** for Animal 201.
>
> Same animal, same formula — the difference is purely the QT measurement method. Robust QT-endpoint detection in mouse ECG is genuinely hard (T wave merges with the J wave, no clean isoelectric return), so this is **deliberately left unresolved** pending input from Dr Kelly-Laubscher / Dr Kelly. Do **not** change the QT method until that discussion has happened.

### Scale (Fix 4)
No animal IDs are hardcoded. The pipeline iterates every `*.txt` file in the data folder. Each file is wrapped in `try/except` so one bad file cannot crash the run.

## Setup — imports and project paths

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

PROJECT_ROOT = Path(r"C:\Users\Vishva\Downloads\ecg_cardiotoxicity_project")
DATA_DIR     = PROJECT_ROOT / "data"
OUTPUT_DIR   = PROJECT_ROOT / "outputs"
FIG_DIR      = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FS                  = 1000
BANDPASS_LOW_HZ     = 0.5
BANDPASS_HIGH_HZ    = 40.0
BANDPASS_ORDER      = 2

MIN_RR_MS           = 60
BEATS_TO_AVERAGE    = 4
PRE_R_MS            = 100
POST_R_MS           = 150

HR_NORMAL_LOW       = 400        # green/red on the bar chart
HR_NORMAL_HIGH      = 500
HR_ACCEPT_LOW       = 300        # used by the R-peak retry logic
HR_ACCEPT_HIGH      = 700

GAP_MIN_S           = 10         # minimum baseline-window length
GAP_MAX_S           = 120        # maximum 'short' baseline-window length
GAP_MAX_LONG_S      = 900        # accept up to 15 min for explicit 'baseline' markers
LONG_USE_S          = 60         # of a long window, only use the first 60 s
SINGLE_FALLBACK_S   = 30         # if only a start marker, use 30 s after it

EXCLUDE_KEYWORDS    = ("oc 1", "oc 2", "oc1", "oc2",
                       "jugular", "jugcan", "injection", "inject")
END_KEYWORDS        = ("end", "done", "stop")
START_HINTS         = ("baseline", "ecg", "start")

print(f"Data folder   : {DATA_DIR}")
print(f"Output folder : {OUTPUT_DIR}")
print(f"Figure folder : {FIG_DIR}")

## Helper functions

In [ ]:
# --- Filename --------------------------------------------------------------
def parse_filename(path: Path):
    """Return (animal_id, recording_date). Handles 4-group names
       (year, month, day, animal) and 3-group names (year, month, animal)."""
    nums = re.findall(r"\d+", path.stem)
    try:
        if len(nums) >= 4:
            y, m, d, a = int(nums[0]), int(nums[1]), int(nums[2]), int(nums[3])
            return a, pd.Timestamp(year=y, month=m, day=d)
        if len(nums) == 3:
            y, m, a = int(nums[0]), int(nums[1]), int(nums[2])
            return a, pd.Timestamp(year=y, month=m, day=1)
    except (ValueError, TypeError):
        pass
    return None, None


# --- Header / column layout (Fix 2) ---------------------------------------
def parse_header_and_layout(path: Path):
    """Read header lines, then the first data row, and figure out:
         n_header   : number of header lines
         lead_cols  : number of leading time/date columns
         ecg_col    : 0-based column index of Channel 3 (the ECG mV channel)
       Lead cols are computed as (n_columns_in_first_data_row - n_channels_in_header)."""
    info = {}
    n_header = 0
    first_data_row = None
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.lstrip()
            if not s or s[0].isdigit() or s[0] in "+-.":
                first_data_row = line.rstrip("\n")
                break
            n_header += 1
            if "=" in line:
                key, _, rest = line.partition("=")
                info[key.strip()] = rest.strip("\n").strip("\t").split("\t")

    titles = [t.strip() for t in info.get("ChannelTitle", []) if t.strip()]
    n_channels = max(len(titles), 1)
    n_cols_first_row = len(first_data_row.split("\t")) if first_data_row else (n_channels + 1)
    lead_cols = max(1, n_cols_first_row - n_channels)

    # Find Channel 3 in the channel-title list; fall back to first/only channel.
    ch3_idx = None
    for i, t in enumerate(titles):
        if t.lower() == "channel 3":
            ch3_idx = i
            break
    if ch3_idx is None:
        ch3_idx = 0
    ecg_col = lead_cols + ch3_idx
    return n_header, lead_cols, ecg_col, titles


# --- Marker detection ------------------------------------------------------
# NOTE: no word boundary (\b) — '*' is non-word, so '\b' would never match #*.
# This matches '#*', '#1', '#2', '#3'.
_MARKER_RE = re.compile(r"#([*1-3])")

def _looks_like_end(text: str)        -> bool:
    low = text.lower()
    return any(k in low for k in END_KEYWORDS)

def _looks_like_excluded(text: str)   -> bool:
    low = text.lower()
    return any(k in low for k in EXCLUDE_KEYWORDS)

def _looks_like_baseline(text: str)   -> bool:
    return "baseline" in text.lower()


# --- File loader -----------------------------------------------------------
def load_ecg_file(path: Path, n_header: int, ecg_col: int):
    """Stream the file. Return:
         voltage     : 1-D float mV array
         markers     : list of (row_index, full_annotation_text, prefix_char)
                       where prefix_char is '*', '1', '2', or '3'."""
    voltages = []
    markers  = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for _ in range(n_header):
            f.readline()
        for line in f:
            stripped = line.rstrip("\n")
            parts = stripped.split("\t")
            if len(parts) <= ecg_col:
                continue
            try:
                v = float(parts[ecg_col])
            except ValueError:
                v = np.nan
            voltages.append(v)
            m = _MARKER_RE.search(stripped)
            if m:
                prefix = m.group(1)              # '*' / '1' / '2' / '3'
                ann_text = stripped[m.start():].strip()
                markers.append((len(voltages) - 1, ann_text, prefix))

    voltage = np.asarray(voltages, dtype=float)
    if np.isnan(voltage).any():
        idx = np.arange(len(voltage))
        good = ~np.isnan(voltage)
        if good.any():
            voltage = np.interp(idx, idx[good], voltage[good])
    return voltage, markers


# --- Baseline window (Fix 1) ----------------------------------------------
def find_baseline_window(markers, n_samples, fs=FS):
    """Tiered search for a baseline (start, end, rule_used, start_text, end_text).
       Tier A: same-prefix pair, gap 10-120 s.
       Tier B: cross-prefix pair, gap 10-120 s.
       Tier C: any pair whose texts both contain 'baseline', gap 10-900 s
               -> use only the first 60 s of that window.
       Tier D: any single start-like marker -> use a 30 s window after it.
       Markers whose text contains procedure keywords (oc, jugular...) are excluded."""
    MIN_GAP   = GAP_MIN_S      * fs
    MAX_GAP   = GAP_MAX_S      * fs
    LONG_GAP  = GAP_MAX_LONG_S * fs
    LONG_USE  = LONG_USE_S     * fs
    FALLBACK  = SINGLE_FALLBACK_S * fs

    tagged = []
    for idx, text, prefix in markers:
        if _looks_like_excluded(text):
            continue
        end_like = _looks_like_end(text)
        tagged.append({"idx": idx, "text": text, "prefix": prefix, "end": end_like})

    starts = [m for m in tagged if not m["end"]]
    ends   = [m for m in tagged if m["end"]]

    def _try(_starts, _ends, lo, hi, label):
        for s in _starts:
            for e in _ends:
                if e["idx"] <= s["idx"]:
                    continue
                gap = e["idx"] - s["idx"]
                if lo <= gap <= hi:
                    return s, e, label
        return None

    # Tier A — same-prefix, short window. '*' first because it's the most common.
    for pre in ("*", "1", "2", "3"):
        result = _try(
            [m for m in starts if m["prefix"] == pre],
            [m for m in ends   if m["prefix"] == pre],
            MIN_GAP, MAX_GAP, f"A_same_{pre}")
        if result:
            s, e, label = result
            return s["idx"], min(e["idx"], n_samples), label, s["text"], e["text"]

    # Tier B — cross-prefix, short window
    result = _try(starts, ends, MIN_GAP, MAX_GAP, "B_cross")
    if result:
        s, e, label = result
        return s["idx"], min(e["idx"], n_samples), label, s["text"], e["text"]

    # Tier C — long baseline window (both markers contain 'baseline')
    bl_starts = [m for m in starts if _looks_like_baseline(m["text"])]
    bl_ends   = [m for m in ends   if _looks_like_baseline(m["text"])]
    result = _try(bl_starts, bl_ends, MIN_GAP, LONG_GAP, "C_long_baseline")
    if result:
        s, e, label = result
        end_idx = min(s["idx"] + LONG_USE, e["idx"], n_samples)
        return s["idx"], end_idx, label, s["text"], e["text"]

    # Tier D — single start-like marker, 30 s fallback
    candidate = None
    for m in starts:
        if any(k in m["text"].lower() for k in START_HINTS):
            candidate = m
            break
    if candidate is None and starts:
        candidate = starts[0]
    if candidate is None and tagged:
        candidate = tagged[0]
    if candidate is not None:
        end_idx = min(candidate["idx"] + FALLBACK, n_samples)
        return candidate["idx"], end_idx, "D_single_30s", candidate["text"], None

    return None, None, "no_markers", None, None


# --- Signal processing ----------------------------------------------------
def bandpass_filter(x, fs=FS, low=BANDPASS_LOW_HZ, high=BANDPASS_HIGH_HZ, order=BANDPASS_ORDER):
    nyq = 0.5 * fs
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    return filtfilt(b, a, x)


def _peaks_with_mad(sig, fs=FS, min_rr_ms=MIN_RR_MS):
    """Median + 4*MAD adaptive threshold (Fix 3)."""
    if sig.size == 0:
        return np.array([], dtype=int), np.nan, np.nan
    med = float(np.median(sig))
    mad = float(np.median(np.abs(sig - med)))
    if mad <= 0:
        return np.array([], dtype=int), med, mad
    height     = med + 4 * mad
    prominence = max(0.3 * mad, 0.005)            # safety floor at 5 uV
    distance   = int(min_rr_ms * fs / 1000)
    peaks, _   = find_peaks(sig, height=height, distance=distance, prominence=prominence)
    return peaks, height, prominence


def detect_r_peaks(sig, fs=FS):
    """Robust R-peak detection. Returns (peaks, inverted_flag, hr_bpm, threshold).
       1. MAD threshold on the signal as-is.
       2. If HR is outside 300-700 bpm, retry on the inverted signal.
       3. Whichever gives an HR closer to the 300-700 band wins."""
    def _hr(peaks):
        if len(peaks) < 2:
            return np.nan
        rr_ms = np.diff(peaks) * (1000.0 / fs)
        return 60000.0 / np.mean(rr_ms)

    pos_peaks, pos_h, _ = _peaks_with_mad(sig, fs)
    pos_hr = _hr(pos_peaks)
    pos_ok = HR_ACCEPT_LOW <= pos_hr <= HR_ACCEPT_HIGH if not np.isnan(pos_hr) else False
    if pos_ok:
        return pos_peaks, False, pos_hr, pos_h

    inv_peaks, inv_h, _ = _peaks_with_mad(-sig, fs)
    inv_hr = _hr(inv_peaks)
    inv_ok = HR_ACCEPT_LOW <= inv_hr <= HR_ACCEPT_HIGH if not np.isnan(inv_hr) else False
    if inv_ok:
        return inv_peaks, True, inv_hr, inv_h

    def _dist(hr):
        if np.isnan(hr):
            return np.inf
        if hr < HR_ACCEPT_LOW:
            return HR_ACCEPT_LOW - hr
        if hr > HR_ACCEPT_HIGH:
            return hr - HR_ACCEPT_HIGH
        return 0.0

    if _dist(inv_hr) < _dist(pos_hr):
        return inv_peaks, True, inv_hr, inv_h
    return pos_peaks, False, pos_hr, pos_h


def average_beats(sig, r_peaks, fs=FS,
                  pre_ms=PRE_R_MS, post_ms=POST_R_MS,
                  group_size=BEATS_TO_AVERAGE):
    pre  = int(pre_ms  * fs / 1000)
    post = int(post_ms * fs / 1000)
    win_len = pre + post
    t_ms = (np.arange(win_len) - pre) * (1000.0 / fs)
    beats = [sig[r - pre: r + post] for r in r_peaks
             if r - pre >= 0 and r + post <= len(sig)]
    if not beats:
        return None, t_ms
    beats = np.vstack(beats)
    n_full_groups = len(beats) // group_size
    if n_full_groups == 0:
        return beats.mean(axis=0), t_ms
    grouped = beats[:n_full_groups * group_size].reshape(n_full_groups, group_size, win_len)
    return grouped.mean(axis=1).mean(axis=0), t_ms


def extract_morphology_features(template, t_ms):
    if template is None:
        return dict.fromkeys(
            ["r_amplitude_mv", "qrs_duration_ms",
             "j_wave_amplitude_mv", "t_wave_amplitude_mv",
             "rt_interval_ms", "qt_ms"], np.nan)
    r_idx = int(np.argmax(template))
    r_amp = float(template[r_idx])
    half = r_amp / 2.0
    left = r_idx
    while left > 0 and template[left] > half:
        left -= 1
    right = r_idx
    while right < len(template) - 1 and template[right] > half:
        right += 1
    qrs_ms = float(t_ms[right] - t_ms[left])
    def window_max(s_ms, e_ms):
        mask = (t_ms >= s_ms) & (t_ms <= e_ms)
        if not mask.any():
            return np.nan, np.nan
        seg = template[mask]; ts = t_ms[mask]
        k = int(np.argmax(seg))
        return float(seg[k]), float(ts[k])
    j_amp, _      = window_max(10, 30)
    t_amp, t_time = window_max(40, 80)
    rt_ms = float(t_time) if not np.isnan(t_time) else np.nan
    baseline = float(np.median(template[t_ms < -70])) if (t_ms < -70).any() else 0.0
    tol = 0.05 * r_amp
    qt_ms = np.nan
    for i in np.where(t_ms >= 40)[0]:
        if abs(template[i] - baseline) <= tol:
            qt_ms = float(t_ms[i])
            break
    return {
        "r_amplitude_mv":      r_amp,
        "qrs_duration_ms":     qrs_ms,
        "j_wave_amplitude_mv": j_amp,
        "t_wave_amplitude_mv": t_amp,
        "rt_interval_ms":      rt_ms,
        "qt_ms":               qt_ms,
    }

## Part 1 — Process every file in the data folder

Every file is wrapped in `try/except`. Each one gets a status and goes into the per-file table at the end.

Status values:
- `OK` — heart rate in 300–700 bpm, used for clustering.
- `NEEDS_REVIEW` — beats were extracted but HR is outside 300–700 bpm. Kept in the audit list, not used for clustering.
- `FAILED_PEAKS` — markers found, segment valid, but fewer than 10 R peaks detected.
- `FAILED_ANNOTATION` — no usable markers in the file.
- `FAILED_COLUMN` — voltage column did not look like ECG (e.g. BPM values).
- `FAILED_FILE` — unexpected exception parsing the file.

In [ ]:
files = sorted(DATA_DIR.glob("*.txt"))
print(f"Found {len(files)} .txt files\n")

# QTc correction formula. Mitchell et al. for mouse ECG (Roisin: this is the
# correct one for mice):
#    QTc = QT / sqrt(RR / 100)
# (QT and RR in milliseconds).
QTC_FORMULA_NAME = "Mitchell"

def mitchell_qtc(qt_ms, rr_ms):
    """QTc = QT / sqrt(RR / 100).  Returns NaN if inputs are invalid."""
    if np.isnan(qt_ms) or np.isnan(rr_ms) or rr_ms <= 0:
        return np.nan
    return float(qt_ms / np.sqrt(rr_ms / 100.0))

per_animal     = []     # used for clustering / templates / features
templates      = {}     # animal_id -> (template, t_ms)
needs_review   = []     # rows with status NEEDS_REVIEW
rows_all       = []     # one dict per file for the per-file table

for path in files:
    row = {
        "file":           path.name,
        "animal_id":      None,
        "beats":          None,
        "HR_bpm":         None,
        "base_start":     None,
        "base_end":       None,
        "duration_s":     None,
        "rule_used":      None,
        "ecg_column":     None,
        "inverted":       False,
        "status":         "",
        "start_annotation": None,
        "end_annotation":   None,
    }
    rows_all.append(row)
    try:
        animal_id, rec_date = parse_filename(path)
        row["animal_id"] = animal_id
        if animal_id is None:
            row["status"] = "FAILED_FILE"
            row["rule_used"] = "filename parse"
            continue

        n_header, lead_cols, ecg_col, titles = parse_header_and_layout(path)
        row["ecg_column"] = ecg_col

        voltage, markers = load_ecg_file(path, n_header, ecg_col)
        if voltage.size == 0:
            row["status"]    = "FAILED_FILE"
            row["rule_used"] = "no voltage rows"
            continue

        # Fix 2 sanity check: ECG voltage should be in roughly +/-15 mV.
        med_abs = float(np.median(np.abs(voltage)))
        if med_abs > 50:    # clearly not mV — almost certainly the BPM column was picked
            row["status"]    = "FAILED_COLUMN"
            row["rule_used"] = f"voltage median |.| = {med_abs:.1f} (looks like BPM)"
            continue

        start, end, rule, s_text, e_text = find_baseline_window(markers, len(voltage))
        row["rule_used"]        = rule
        row["start_annotation"] = s_text
        row["end_annotation"]   = e_text
        if start is None:
            row["status"] = "FAILED_ANNOTATION"
            continue

        row["base_start"] = start
        row["base_end"]   = end
        row["duration_s"] = (end - start) / FS

        segment = voltage[start:end]
        if segment.size < 2 * FS:
            row["status"] = "FAILED_ANNOTATION"
            row["rule_used"] = f"{rule} (window too short: {segment.size} samples)"
            continue

        filtered = bandpass_filter(segment)
        peaks, inverted, hr, _ = detect_r_peaks(filtered)
        row["inverted"] = inverted
        if inverted:
            filtered = -filtered

        if len(peaks) < 10:
            row["status"] = "FAILED_PEAKS"
            row["beats"]  = int(len(peaks))
            row["HR_bpm"] = float(hr) if not np.isnan(hr) else None
            continue

        rr_ms          = np.diff(peaks) * (1000.0 / FS)
        template, t_ms = average_beats(filtered, peaks)
        feats          = extract_morphology_features(template, t_ms)

        # QTc via Mitchell formula:  QTc = QT / sqrt(RR/100)
        rr_mean_ms = float(np.mean(rr_ms))
        feats["qtc_ms"]      = mitchell_qtc(feats["qt_ms"], rr_mean_ms)
        feats["qtc_formula"] = QTC_FORMULA_NAME

        row["beats"]  = int(len(peaks))
        row["HR_bpm"] = float(hr)

        record = {
            "animal_id":      animal_id,
            "recording_date": rec_date,
            "source_file":    path.name,
            "n_beats":        int(len(peaks)),
            "heart_rate_bpm": float(hr),
            "rr_mean_ms":     rr_mean_ms,
            "rr_std_ms":      float(np.std(rr_ms, ddof=1)) if len(rr_ms) > 1 else 0.0,
            "rr_intervals_ms": rr_ms,
            **feats,
        }

        if HR_ACCEPT_LOW <= hr <= HR_ACCEPT_HIGH:
            row["status"] = "OK"
            per_animal.append(record)
            templates[animal_id] = (template, t_ms)
        else:
            row["status"] = "NEEDS_REVIEW"
            needs_review.append(record)

    except Exception as e:
        row["status"]    = "FAILED_FILE"
        row["rule_used"] = f"{type(e).__name__}: {e}"

print(f"QTc formula     : {QTC_FORMULA_NAME}   [QTc = QT / sqrt(RR/100)]")
print(f"OK              : {sum(1 for r in rows_all if r['status']=='OK')}")
print(f"NEEDS_REVIEW    : {sum(1 for r in rows_all if r['status']=='NEEDS_REVIEW')}")
print(f"FAILED_PEAKS    : {sum(1 for r in rows_all if r['status']=='FAILED_PEAKS')}")
print(f"FAILED_ANNOTATION: {sum(1 for r in rows_all if r['status']=='FAILED_ANNOTATION')}")
print(f"FAILED_COLUMN   : {sum(1 for r in rows_all if r['status']=='FAILED_COLUMN')}")
print(f"FAILED_FILE     : {sum(1 for r in rows_all if r['status']=='FAILED_FILE')}")

# --- Verification of the QTc formula on Animal 201 ------------------------
a201 = next((a for a in (per_animal + needs_review) if a["animal_id"] == 201), None)
if a201 is not None:
    print("\nQTc formula verification (Animal 201):")
    print(f"  QT = {a201['qt_ms']:.1f} ms,  RR = {a201['rr_mean_ms']:.1f} ms")
    print(f"  QTc = QT / sqrt(RR/100)")
    print(f"      = {a201['qt_ms']:.1f} / sqrt({a201['rr_mean_ms']:.1f}/100)")
    print(f"      = {a201['qt_ms']:.1f} / {np.sqrt(a201['rr_mean_ms']/100):.3f}")
    print(f"      = {a201['qtc_ms']:.1f} ms")

### Per-file table (every input file, success or skip)

Saved to `outputs/per_file_audit.csv` so Roisin can sort and inspect failures.

In [ ]:
audit_df = pd.DataFrame(rows_all)[[
    "animal_id", "file", "status", "beats", "HR_bpm",
    "base_start", "base_end", "duration_s",
    "rule_used", "ecg_column", "inverted",
    "start_annotation", "end_annotation",
]].copy()
audit_df = audit_df.sort_values(["status", "animal_id"], na_position="last").reset_index(drop=True)

audit_path = OUTPUT_DIR / "per_file_audit.csv"
audit_df.to_csv(audit_path, index=False)
print(f"Saved per-file audit to: {audit_path}\n")

with pd.option_context("display.max_rows", 200,
                       "display.max_colwidth", 50,
                       "display.width", 220):
    print(audit_df.to_string(index=False))

### Recording dates — acute vs chronic

In [ ]:
if not per_animal:
    print("No animals processed successfully — cannot guess study groups.")
else:
    unique_dates = sorted({a["recording_date"].date() for a in per_animal})
    if len(unique_dates) >= 2:
        gaps = [(unique_dates[i + 1] - unique_dates[i]).days for i in range(len(unique_dates) - 1)]
        split_at = int(np.argmax(gaps))
        early_dates = set(unique_dates[: split_at + 1])
        study_label = {d: ("acute" if d in early_dates else "chronic") for d in unique_dates}
        print(f"Largest date gap = {max(gaps)} days -> splitting at {unique_dates[split_at]}")
    else:
        study_label = {d: "unknown" for d in unique_dates}
    for a in per_animal:
        a["study_guess"] = study_label.get(a["recording_date"].date(), "unknown")
    grp_counts = pd.Series([a["study_guess"] for a in per_animal]).value_counts()
    print("\nAnimals per study group:")
    print(grp_counts.to_string())

## Part 2 — Visualise all animals

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
cmap = plt.cm.tab20
ids_sorted = sorted(templates.keys())
show_legend = len(ids_sorted) <= 20
for i, aid in enumerate(ids_sorted):
    tmpl, t_ms = templates[aid]
    ax.plot(t_ms, tmpl, color=cmap(i % 20), linewidth=0.8,
            alpha=0.7, label=f"Animal {aid}" if show_legend else None)
ax.axvline(0, color="k", linewidth=0.5, linestyle="--", alpha=0.5)
ax.set_xlabel("Time relative to R peak (ms)")
ax.set_ylabel("Voltage (mV)")
ax.set_title(f"Averaged Beat Template — {len(ids_sorted)} Animals")
if show_legend:
    ax.legend(loc="upper right", ncol=2, fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "07_all_averaged_beats.png", dpi=150)
plt.show()

In [ ]:
df_hr = pd.DataFrame(per_animal).sort_values("animal_id") if per_animal else pd.DataFrame()
if not df_hr.empty:
    colors = ["green" if HR_NORMAL_LOW <= hr <= HR_NORMAL_HIGH else "orange"
              for hr in df_hr["heart_rate_bpm"]]
    fig, ax = plt.subplots(figsize=(max(12, 0.18 * len(df_hr)), 5))
    ax.bar(df_hr["animal_id"].astype(str), df_hr["heart_rate_bpm"], color=colors, edgecolor="black")
    ax.axhline(HR_NORMAL_LOW,  color="gray", linestyle="--", linewidth=0.8)
    ax.axhline(HR_NORMAL_HIGH, color="gray", linestyle="--", linewidth=0.8)
    ax.set_xlabel("Animal ID")
    ax.set_ylabel("Heart rate (bpm)")
    ax.set_title(f"Heart Rate per Animal (green = {HR_NORMAL_LOW}-{HR_NORMAL_HIGH} bpm,"
                 f" orange = within 300-700 acceptance band)")
    ax.tick_params(axis="x", labelrotation=90)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "08_heart_rates.png", dpi=150)
    plt.show()
else:
    print("No animals to plot.")

In [ ]:
if per_animal:
    n = len(per_animal)
    ncols = 6 if n > 24 else 4
    nrows = max(1, int(np.ceil(n / ncols)))
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.6 * ncols, 2.2 * nrows), sharey=True)
    axes = np.array(axes).reshape(-1)
    sa = sorted(per_animal, key=lambda x: x["animal_id"])
    for i, a in enumerate(sa):
        ax = axes[i]
        ax.plot(a["rr_intervals_ms"], linewidth=0.5)
        ax.set_title(f"A{a['animal_id']} {a['heart_rate_bpm']:.0f}bpm", fontsize=7)
        ax.tick_params(labelsize=6)
        ax.grid(alpha=0.3)
    for j in range(len(sa), len(axes)):
        axes[j].axis("off")
    fig.suptitle("RR Intervals per Animal", y=1.01, fontsize=11)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "09_rr_all_animals.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No animals to plot.")

## Part 3 — Feature table

In [ ]:
feature_cols = [
    "animal_id", "recording_date", "study_guess", "source_file",
    "n_beats", "heart_rate_bpm", "rr_mean_ms", "rr_std_ms",
    "r_amplitude_mv", "qrs_duration_ms",
    "j_wave_amplitude_mv", "t_wave_amplitude_mv",
    "rt_interval_ms", "qt_ms", "qtc_ms", "qtc_formula",
]
df = pd.DataFrame([{k: a.get(k) for k in feature_cols} for a in per_animal])
if not df.empty:
    df = df.sort_values("animal_id").reset_index(drop=True)
df.head(15) if not df.empty else "(empty)"

## Part 4 — Clustering

In [ ]:
if df.empty:
    print("No animals to cluster.")
else:
    cluster_feature_cols = [
        "heart_rate_bpm", "rr_mean_ms", "rr_std_ms",
        "r_amplitude_mv", "qrs_duration_ms",
        "j_wave_amplitude_mv", "t_wave_amplitude_mv",
        "qt_ms", "qtc_ms",
    ]
    X_raw    = df[cluster_feature_cols].copy()
    X_filled = X_raw.fillna(X_raw.median(numeric_only=True))
    X_scaled = StandardScaler().fit_transform(X_filled)

    def run_kmeans(k):
        if len(df) < k:
            return np.full(len(df), -1)
        return KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_scaled)

    df["cluster_k2"] = run_kmeans(2)
    df["cluster_k5"] = run_kmeans(5)

    print("k=2 cluster sizes:", dict(df["cluster_k2"].value_counts()))
    print("k=5 cluster sizes:", dict(df["cluster_k5"].value_counts()))
    if "study_guess" in df.columns:
        print("\ncluster_k2 vs study_guess:")
        print(pd.crosstab(df["cluster_k2"], df["study_guess"]))

In [ ]:
if not df.empty and templates:
    clusters_present = sorted(c for c in df["cluster_k5"].unique() if c >= 0)
    n_panels = max(len(clusters_present), 1)
    fig, axes = plt.subplots(1, n_panels, figsize=(3.5 * n_panels, 4),
                             sharey=True, squeeze=False)
    axes = axes[0]
    last_t_ms = None
    for ax, c in zip(axes, clusters_present):
        members = df.loc[df["cluster_k5"] == c, "animal_id"].tolist()
        stack = []
        for aid in members:
            if aid in templates:
                tmpl, t_ms = templates[aid]
                ax.plot(t_ms, tmpl, color="gray", alpha=0.4, linewidth=0.7)
                stack.append(tmpl); last_t_ms = t_ms
        if stack and last_t_ms is not None:
            centroid = np.mean(np.vstack(stack), axis=0)
            ax.plot(last_t_ms, centroid, color="red", linewidth=2.0, label="cluster mean")
        ax.set_title(f"k=5 cluster {c}  (n={len(members)})", fontsize=9)
        ax.set_xlabel("Time from R peak (ms)")
        ax.axvline(0, color="k", linewidth=0.5, linestyle="--", alpha=0.5)
        ax.grid(alpha=0.3); ax.legend(fontsize=7)
    axes[0].set_ylabel("Voltage (mV)")
    fig.suptitle("Averaged-beat templates by k=5 cluster", y=1.02)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "10_cluster_beats.png", dpi=150, bbox_inches="tight")
    plt.show()

## Part 5 — Master CSV

In [ ]:
if not df.empty:
    out_cols = [
        "animal_id", "recording_date", "study_guess",
        "n_beats", "heart_rate_bpm", "rr_mean_ms", "rr_std_ms",
        "r_amplitude_mv", "qrs_duration_ms",
        "j_wave_amplitude_mv", "t_wave_amplitude_mv",
        "qt_ms", "qtc_ms", "qtc_formula",
        "cluster_k2", "cluster_k5",
    ]
    summary = df[out_cols].copy()
    summary["recording_date"] = pd.to_datetime(summary["recording_date"]).dt.date
    csv_path = OUTPUT_DIR / "all_animals_features.csv"
    summary.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path}")
    print(f"Rows : {len(summary)}")
    summary.head(20)
else:
    print("No animals to save.")

## Final summary

In [ ]:
print("=" * 60)
print("NOTEBOOK 02 — SUMMARY")
print("=" * 60)
status_counts = pd.Series([r["status"] for r in rows_all]).value_counts()
print(f"Total files found      : {len(files)}")
for status, n in status_counts.items():
    print(f"  {status:<20} : {n}")

if per_animal:
    hrs = pd.Series([a["heart_rate_bpm"] for a in per_animal])
    print(f"\nHR (bpm)        : min {hrs.min():.0f}   mean {hrs.mean():.0f}   max {hrs.max():.0f}")
    print(f"In 400-500 bpm  : {((hrs >= HR_NORMAL_LOW) & (hrs <= HR_NORMAL_HIGH)).sum()}/{len(hrs)}")

if needs_review:
    print("\nNeeds manual review (HR outside 300-700 bpm):")
    for a in needs_review:
        print(f"   - animal {a['animal_id']:>4}  HR {a['heart_rate_bpm']:6.1f} bpm  ({a['source_file']})")

failed = [r for r in rows_all if r["status"].startswith("FAILED")]
if failed:
    print("\nFailed files (ask Roisin about these):")
    for r in failed[:40]:
        print(f"   - {r['file']}: {r['status']} — {r['rule_used']}")
    if len(failed) > 40:
        print(f"   ... and {len(failed) - 40} more (see outputs/per_file_audit.csv)")

print("\nFigures saved to:", FIG_DIR)
print("Feature CSV     :", OUTPUT_DIR / "all_animals_features.csv")
print("Per-file audit  :", OUTPUT_DIR / "per_file_audit.csv")